In [2]:
import os
import re
import csv
from collections import defaultdict
from statistics import mean

def parse_log_file_last_metrics(path):
    """从 log 文件中提取最后一次 eval 的指标（支持科学计数法）"""
    eval_metrics = {}
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()

    i = len(lines) - 1
    while i >= 0:
        if lines[i].strip().startswith("start to eval"):
            # hit20
            if i + 1 < len(lines) and lines[i + 1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.+-eE]+)", lines[i + 1])
                if m:
                    eval_metrics["hit20"] = float(m.group(1))
            # hit50
            if i + 2 < len(lines) and lines[i + 2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.+-eE]+)", lines[i + 2])
                if m:
                    eval_metrics["hit50"] = float(m.group(1))
            # hit100
            if i + 3 < len(lines) and lines[i + 3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.+-eE]+)", lines[i + 3])
                if m:
                    eval_metrics["hit100"] = float(m.group(1))
            # roc_auc, pr_auc, f1, mrr
            if i + 4 < len(lines) and lines[i + 4].startswith("roc_auc"):
                nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", lines[i + 4])
                if len(nums) >= 4:
                    fields = ['roc_auc', 'pr_auc', 'f1', 'mrr']
                    eval_metrics.update(dict(zip(fields, map(float, nums[-4:]))))
            if eval_metrics:
                break
        i -= 1
    return eval_metrics


# ==== 参数 ====
dataset = "cora_ml"
ratio = 0.9
epoch = 75
seed = 1
cs = [1]
pos_ratios = [0.2, 1, 5.0, 10]
hs = [12, 13, 14, 15]
gammas = [2, 3, 4, 5, 6, 7]
alphas = [0.1, 0.5, 1, 5, 10, 15, 24, 36, 50, 74, 100, 120, 150, 200, 250, 300]

log_base_dir = "."
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

ALL_METRICS = ['hit20', 'hit50', 'hit100', 'roc_auc', 'pr_auc', 'f1', 'mrr']

metric_bucket = defaultdict(list)
metric_keys = set()

# ==== 解析日志 ====
for pos_ratio in pos_ratios:
    for c in cs:
        for h in hs:
            for gamma in gammas:
                for alpha in alphas:
                    fname = f"s{seed}-h{h}-c{c}-pr{pos_ratio}-r{ratio}-g{gamma}-a{alpha}.log"
                    log_path = os.path.join(log_base_dir, fname)
                    key = (pos_ratio, c, h, gamma, alpha)

                    if not os.path.isfile(log_path):
                        metrics = {k: 0.0 for k in ALL_METRICS}
                    else:
                        metrics = parse_log_file_last_metrics(log_path)
                        if not metrics:
                            metrics = {k: 0.0 for k in ALL_METRICS}
                    metric_bucket[key].append(metrics)
                    metric_keys.update(metrics.keys())

# ==== 写入 CSV ====
csv_path = os.path.join(output_dir, f"{dataset}_grid_result.csv")
metric_keys = sorted(metric_keys)
header = ["pos_ratio", "c", "h", "gamma", "alpha"] + metric_keys

with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    for key, metrics_list in metric_bucket.items():
        pos_ratio, c, h, gamma, alpha = key
        row = {"pos_ratio": pos_ratio, "c": c, "h": h, "gamma": gamma, "alpha": alpha}
        for k in metric_keys:
            vals = [m[k] for m in metrics_list if k in m]
            mu = round(mean(vals), 6) if vals else 0.0
            row[k] = mu
        writer.writerow(row)

print(f"[INFO] 写入完毕: {csv_path}")


[INFO] 写入完毕: ./results/cora_ml_grid_result.csv


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact

# === 读取结果 ===
csv_path = "./results/cora_ml_grid_result.csv"
df = pd.read_csv(csv_path)

# 可选指标
metric_options = ["hit20", "hit50", "hit100", "roc_auc", "pr_auc", "f1", "mrr"]
pos_ratio_options = sorted(df["pos_ratio"].unique())

def plot_metric(metric="roc_auc", pos_ratio=1.0, ylim_min=None, ylim_max=None):
    sns.set(style="whitegrid")
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
    axes = axes.flatten()

    # 颜色映射：gamma 越大颜色越深
    gammas = sorted(df["gamma"].unique())
    palette = sns.color_palette("Blues", n_colors=len(gammas))

    h_values = sorted(df["h"].unique())
    valid_data = df[df["pos_ratio"] == pos_ratio]

    # 确定统一的 y 轴范围
    y_vals = valid_data[metric]
    y_nonzero = y_vals[y_vals > 0]
    if len(y_nonzero) == 0:
        print("所有数据为 0，无法绘制。")
        return
    ymin_auto, ymax_auto = y_nonzero.min(), y_nonzero.max()
    ymin = ylim_min if ylim_min is not None else ymin_auto
    ymax = ylim_max if ylim_max is not None else ymax_auto

    for idx, h in enumerate(h_values):
        ax = axes[idx]
        sub = valid_data[valid_data["h"] == h]
        for j, gamma in enumerate(gammas):
            d = sub[sub["gamma"] == gamma].sort_values("alpha")
            ax.plot(d["alpha"], d[metric], marker="o", color=palette[j], label=f"γ={gamma}")
        ax.set_title(f"h={h}")
        ax.set_xlabel("alpha")
        ax.grid(True)
        ax.set_ylim(ymin, ymax)
        if idx == 0:
            ax.set_ylabel(metric)
        ax.legend(title="gamma", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.suptitle(f"{metric} vs alpha (pos_ratio={pos_ratio})", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# === 交互式控件 ===
interact(
    plot_metric,
    metric=widgets.Dropdown(options=metric_options, value="roc_auc", description="Metric"),
    pos_ratio=widgets.Dropdown(options=pos_ratio_options, value=1.0, description="pos_ratio"),
    ylim_min=widgets.FloatText(value=None, description="ymin (可留空)"),
    ylim_max=widgets.FloatText(value=None, description="ymax (可留空)")
)


interactive(children=(Dropdown(description='Metric', index=3, options=('hit20', 'hit50', 'hit100', 'roc_auc', …

<function __main__.plot_metric(metric='roc_auc', pos_ratio=1.0, ylim_min=None, ylim_max=None)>